# 🌋 LLaVA Training - Colab Optimized

This notebook provides a streamlined workflow to train LLaVA on the MVTec dataset using Google Colab Pro (A100 recommended).

### 📋 Prerequisites
1.  **Google Drive**: Ensure your data is in `/content/drive/MyDrive/Project_Data`.
2.  **GPU**: Runtime > Change runtime type > A100 GPU.

In [ ]:
# @title 1. Initialize Environment & Mount Drive
from google.colab import drive
import os

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Define Paths
PROJECT_DIR = '/content/drive/MyDrive/Project_Data'
LLAVA_DIR = os.path.join(PROJECT_DIR, 'LLaVA')

# 3. Verify Directory
if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(f"❌ Project directory not found: {PROJECT_DIR}\nPlease ensure your Google Drive folder structure is correct.")

os.chdir(PROJECT_DIR)
print(f"✅ Working Directory: {os.getcwd()}")

In [ ]:
# @title 2. Install Dependencies (Clean Install)
# We install specific versions to avoid the 'torchvision' conflict.

print("⏳ Installing dependencies... (This may take 2-3 minutes)")

# 1. Uninstall potential conflicts
!pip uninstall -y torch torchvision ORT 

# 2. Install Core specific versions compatible with Colab's CUDA
# Using Torch 2.1.2 + Vision 0.16.2 which is very stable on Colab
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121

# 3. Clone LLaVA if needed
if not os.path.exists('LLaVA'):
    print("⬇️ Cloning LLaVA repository...")
    !git clone https://github.com/haotian-liu/LLaVA.git

# 4. Install other requirements
# We install LLaVA dependencies manually to avoid the bad 'torchvision' requirement in its setup.py
!pip install transformers==4.37.2 accelerate==0.28.0 peft==0.9.0 bitsandbytes==0.41.0 gradio==4.16.0 shortuuid
!pip install einops-exts einops container_abcs uvicorn pydantic==1.10.13 markdown2 scikit-learn
!pip install flash-attn --no-build-isolation

# 5. Install LLaVA (No Dependencies mode to keep our good versions)
os.chdir(LLAVA_DIR)
!pip install -e . --no-deps
os.chdir(PROJECT_DIR)

print("✅ Setup Complete.")

In [ ]:
# @title 3. Verification Check
import torch
import llava

print("="*40)
print(f"🐍 PyTorch: {torch.__version__}")
try:
    import torchvision
    print(f"🖼️ TorchVision: {torchvision.__version__}")
except ImportError:
    print("🖼️ TorchVision: Not Installed")

if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("❌ No GPU detected! Check Runtime settings.")

print("✅ LLaVA imported successfully.")
print("="*40)

In [ ]:
# @title 4. Run Training
# Arguments configured for MVTec dataset

!python LLaVA/llava/train/train_mem.py \
    --model_name_or_path liuhaotian/llava-v1.5-7b \
    --version v1 \
    --data_path ./mvtec_train.json \
    --image_folder ./mvtec_anomaly_detection \
    --vision_tower openai/clip-vit-large-patch14-336 \
    --mm_projector_type mlp2x_gelu \
    --mm_vision_select_layer -2 \
    --mm_use_im_start_end False \
    --mm_use_im_patch_token False \
    --image_aspect_ratio pad \
    --group_by_modality_length True \
    --bf16 True \
    --output_dir ./checkpoints/llava-mvtec-lora \
    --num_train_epochs 3 \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 4 \
    --gradient_accumulation_steps 1 \
    --evaluation_strategy "no" \
    --save_strategy "steps" \
    --save_steps 50 \
    --save_total_limit 2 \
    --learning_rate 2e-4 \
    --weight_decay 0.0 \
    --warmup_ratio 0.03 \
    --lr_scheduler_type "cosine" \
    --logging_steps 1 \
    --tf32 True \
    --model_max_length 2048 \
    --gradient_checkpointing True \
    --dataloader_num_workers 4 \
    --lazy_preprocess True \
    --report_to none \
    --lora_enable True \
    --lora_r 128 \
    --lora_alpha 256 \
    --lora_dropout 0.05